# Car Insurance Batch Inference & Transform Pipeline

This notebook:
1. Generates synthetic test data for the car insurance pricing model
2. Runs batch predictions via the SPCS inference service (`predict`)
3. Queries the inference table to inspect logged predictions
4. Creates a stream + task pipeline that monitors new inference records and runs `transform` on them every 5 minutes

## 1. Setup and Configuration

In [96]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F

session = Session.builder.config(
    "connection_name",
    os.getenv("SNOWFLAKE_CONNECTION_NAME") or "keypair"
).create()

In [97]:
DATABASE = "CC_ML_INSURANCE"
SCHEMA = "CAR_PRICING"
MODEL_NAME = "CAR_INSURANCE_PRICING_MODEL"
MODEL_VERSION = "V3"
SERVICE_NAME = "CAR_INSURANCE_INFERENCE_SVC"

session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected to: {session.get_current_account()}")
print(f"Using: {DATABASE}.{SCHEMA}")
print(f"Model: {MODEL_NAME} version {MODEL_VERSION}")
print(f"Service: {SERVICE_NAME}")

Connected to: "phb14991"
Using: CC_ML_INSURANCE.CAR_PRICING
Model: CAR_INSURANCE_PRICING_MODEL version V3
Service: CAR_INSURANCE_INFERENCE_SVC


## 2. Generate Synthetic Test Data

In [98]:
np.random.seed(99)
random.seed(99)

N_TEST = 200

CAR_MAKES = {
    'Toyota': ['Camry', 'Corolla', 'RAV4', 'Highlander', 'Prius'],
    'Honda': ['Civic', 'Accord', 'CR-V', 'Pilot', 'Odyssey'],
    'Ford': ['F-150', 'Mustang', 'Explorer', 'Escape', 'Bronco'],
    'BMW': ['3 Series', '5 Series', 'X3', 'X5', 'M3'],
    'Mercedes': ['C-Class', 'E-Class', 'GLC', 'GLE', 'S-Class'],
    'Chevrolet': ['Silverado', 'Malibu', 'Equinox', 'Tahoe', 'Corvette'],
    'Tesla': ['Model 3', 'Model Y', 'Model S', 'Model X'],
    'Nissan': ['Altima', 'Rogue', 'Sentra', 'Pathfinder', 'Maxima']
}

FUEL_TYPES = ['Gasoline', 'Diesel', 'Hybrid', 'Electric']
TRANSMISSIONS = ['Automatic', 'Manual', 'CVT']
COVERAGE_TYPES = ['Basic', 'Standard', 'Premium', 'Comprehensive']
STATES = ['CA', 'TX', 'FL', 'NY', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI']

current_year = datetime.now().year

# Fetch existing customer IDs from the CUSTOMERS table
existing_customers = session.table(f"{DATABASE}.{SCHEMA}.CUSTOMERS").select("CUSTOMER_ID").to_pandas()
customer_id_list = existing_customers["CUSTOMER_ID"].tolist()
print(f"Loaded {len(customer_id_list)} existing customer IDs from {DATABASE}.{SCHEMA}.CUSTOMERS")

# Generate only policy-level (spine) data — customer features will come from the Feature Store
test_records = []
for i in range(N_TEST):
    customer_id = random.choice(customer_id_list)

    car_make = random.choice(list(CAR_MAKES.keys()))
    car_model = random.choice(CAR_MAKES[car_make])
    car_year = random.randint(2010, current_year)
    car_age = current_year - car_year
    kilometers = int(max(1000, car_age * np.random.normal(15000, 5000) + np.random.normal(0, 5000)))
    engine_size = random.choice([1.5, 1.8, 2.0, 2.4, 2.5, 3.0, 3.5, 4.0, 5.0])

    if car_make == 'Tesla':
        fuel_type = 'Electric'
    else:
        fuel_type = random.choices(FUEL_TYPES, weights=[0.7, 0.1, 0.15, 0.05])[0]

    transmission = random.choices(TRANSMISSIONS, weights=[0.7, 0.15, 0.15])[0]
    coverage_type = random.choice(COVERAGE_TYPES)

    base_price = {'Toyota': 28000, 'Honda': 27000, 'Ford': 35000, 'BMW': 55000,
                  'Mercedes': 60000, 'Chevrolet': 32000, 'Tesla': 50000, 'Nissan': 26000}[car_make]
    depreciation = 0.85 ** car_age
    km_factor = max(0.5, 1 - (kilometers / 300000))
    estimated_car_value = round(base_price * depreciation * km_factor, 2)

    test_records.append({
        'CUSTOMER_ID': customer_id,
        'CAR_MAKE': car_make,
        'CAR_MODEL': car_model,
        'CAR_AGE': car_age,
        'KILOMETERS': kilometers,
        'ENGINE_SIZE': engine_size,
        'FUEL_TYPE': fuel_type,
        'TRANSMISSION': transmission,
        'COVERAGE_TYPE': coverage_type,
        'ESTIMATED_CAR_VALUE': estimated_car_value,
        'GENDER': random.choice(['M', 'F']),
        'STATE': random.choice(STATES)
    })

test_df = pd.DataFrame(test_records)
print(f"Generated {len(test_df)} synthetic test records (policy-level spine only)")
test_df.head()

Loaded 5000 existing customer IDs from CC_ML_INSURANCE.CAR_PRICING.CUSTOMERS
Generated 200 synthetic test records (policy-level spine only)


,CUSTOMER_ID,CAR_MAKE,CAR_MODEL,CAR_AGE,KILOMETERS,ENGINE_SIZE,FUEL_TYPE,TRANSMISSION,COVERAGE_TYPE,ESTIMATED_CAR_VALUE,GENDER,STATE
0,CUST_003310,Tesla,Model Y,11,167456,2.4,Electric,Automatic,Basic,4183.58,F,OH
1,CUST_004349,Honda,Odyssey,1,23065,2.4,Gasoline,Automatic,Standard,21185.53,F,OH
2,CUST_001742,Ford,Escape,11,156150,3.0,Gasoline,Manual,Basic,2928.51,F,PA
3,CUST_003299,Toyota,RAV4,16,304542,3.5,Gasoline,CVT,Basic,1039.52,F,NC
4,CUST_001623,Nissan,Altima,5,60334,2.0,Gasoline,CVT,Standard,9216.23,F,MI


In [99]:
# Upload test data to Snowflake
session.write_pandas(test_df, "TEST_BATCH_DATA", auto_create_table=True, overwrite=True)
print(f"Uploaded {session.table('TEST_BATCH_DATA').count()} rows to {DATABASE}.{SCHEMA}.TEST_BATCH_DATA")

Uploaded 200 rows to CC_ML_INSURANCE.CAR_PRICING.TEST_BATCH_DATA


## 2b. Enrich Test Data with Feature Store

Use the Snowflake Feature Store to retrieve customer-level features (risk score, credit info, policy aggregates, etc.) for the existing customers in our test batch. This mirrors how the training dataset was built in `car_insurance_ml.ipynb`.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

# Connect to the existing Feature Store
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

# Retrieve the registered Feature View
customer_fv = fs.get_feature_view("CUSTOMER_RISK_FEATURES", "v1")
print(f"Feature View: {customer_fv.name} v{customer_fv.version}")

# Build the spine DataFrame from the uploaded test data
# The spine needs CUSTOMER_ID (join key) + a timestamp for point-in-time lookup
spine_df = (
    session.table("TEST_BATCH_DATA")
    .select(
        "CUSTOMER_ID",
        "CAR_MAKE", "CAR_MODEL", "CAR_AGE", "KILOMETERS", "ENGINE_SIZE",
        "FUEL_TYPE", "TRANSMISSION", "COVERAGE_TYPE", "ESTIMATED_CAR_VALUE",
        "GENDER", "STATE"
    )
    .with_column("TS", F.current_timestamp())
)

# Use the Feature Store to enrich the spine with customer-level features
# retrieve_feature_values returns a Snowpark DataFrame directly
enriched_df = fs.retrieve_feature_values(
    spine_df=spine_df,
    features=[customer_fv],
    spine_timestamp_col="TS"
)
print(f"Enriched dataset rows: {enriched_df.count()}")
print(f"Enriched columns: {enriched_df.columns}")
enriched_df.show(5)

## 3. Load Model & Run Batch Predictions via Service

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
mv = registry.get_model(MODEL_NAME).version(MODEL_VERSION)

print(f"Loaded model: {mv.model_name} version {mv.version_name}")
print(f"Functions: {mv.show_functions()}")

Loaded model: CAR_INSURANCE_PRICING_MODEL version V3
Functions: [{'name': 'PREDICT', 'target_method': 'predict', 'target_method_function_type': 'FUNCTION', 'signature': ModelSignature(
                    inputs=[
                        FeatureSpec(dtype=DataType.INT64, name='CAR_AGE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='KILOMETERS', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='ENGINE_SIZE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='ESTIMATED_CAR_VALUE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='AGE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='YEARS_LICENSED', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='CLAIMS_HISTORY', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='CREDIT_SCORE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='RISK_SCORE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='AVG_CLAIMS_PER_YEAR', nullable=True),
		FeatureSpec(dtype=DataType.IN

In [ ]:
input_cols = [
    'CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
    'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
    'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE',
    'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE',
    'CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE',
    'GENDER', 'STATE'
]

# Use the Feature Store-enriched DataFrame instead of raw TEST_BATCH_DATA
batch_input_df = enriched_df.select(input_cols)

# Run predictions via the SPCS service (compute pool, not warehouse)
predictions = mv.run(
    batch_input_df,
    function_name="predict",
    service_name=SERVICE_NAME
)

print(f"Batch predictions via SPCS service ({SERVICE_NAME}):")
print(f"Rows predicted: {predictions.count()}")
predictions.show(10)

## 4. Query the Inference Table

The inference table is auto-populated when `autocapture=True` was set during `create_service()`.
It logs every prediction request with timestamps, query IDs, latency, and input/output data.

In [ ]:
inference_table_ref = f"'{DATABASE}.{SCHEMA}.{MODEL_NAME}'"

inference_df = session.sql(f"""
    SELECT
        TIMESTAMP,
        RESOURCE_ATTRIBUTES:"snow.query.id"::STRING AS QUERY_ID,
        RESOURCE_ATTRIBUTES:"snow.service.name"::STRING AS SERVICE_NAME,
        RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING AS FUNCTION_NAME,
        OBJECT_CONSTRUCT(
            'AGE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE",
            'CAR_MAKE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MAKE",
            'CAR_MODEL', RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MODEL",
            'CAR_AGE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_AGE",
            'KILOMETERS', RECORD_ATTRIBUTES:"snow.model_serving.request.data.KILOMETERS",
            'ENGINE_SIZE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.ENGINE_SIZE",
            'ESTIMATED_CAR_VALUE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.ESTIMATED_CAR_VALUE",
            'FUEL_TYPE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.FUEL_TYPE",
            'TRANSMISSION', RECORD_ATTRIBUTES:"snow.model_serving.request.data.TRANSMISSION",
            'COVERAGE_TYPE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.COVERAGE_TYPE",
            'CREDIT_SCORE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.CREDIT_SCORE",
            'CLAIMS_HISTORY', RECORD_ATTRIBUTES:"snow.model_serving.request.data.CLAIMS_HISTORY",
            'YEARS_LICENSED', RECORD_ATTRIBUTES:"snow.model_serving.request.data.YEARS_LICENSED",
            'RISK_SCORE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.RISK_SCORE",
            'AVG_CLAIMS_PER_YEAR', RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CLAIMS_PER_YEAR",
            'TOTAL_POLICIES', RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_POLICIES",
            'AVG_CAR_AGE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CAR_AGE",
            'AVG_KILOMETERS', RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_KILOMETERS",
            'TOTAL_CAR_VALUE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_CAR_VALUE",
            'AVG_DEDUCTIBLE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_DEDUCTIBLE",
            'GENDER', RECORD_ATTRIBUTES:"snow.model_serving.request.data.GENDER",
            'STATE', RECORD_ATTRIBUTES:"snow.model_serving.request.data.STATE"
        ) AS INPUT_DATA,
        RECORD_ATTRIBUTES:"snow.model_serving.response.data.PREDICTED_PREMIUM" AS PREDICTED_PREMIUM
    FROM TABLE(INFERENCE_TABLE({inference_table_ref}))
    ORDER BY TIMESTAMP DESC
    LIMIT 20
""")

print(f"Inference table records for {MODEL_NAME}:")
inference_df.show()

Inference table records for CAR_INSURANCE_PRICING_MODEL:
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TIMESTAMP"                 |"QUERY_ID"                            |"SERVICE_NAME"               |"FUNCTION_NAME"  |"INPUT_DATA"                         |"PREDICTED_PREMIUM"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-03-03 16:47:39.786160  |01c2c846-0207-f706-006a-44071151b10a  |CAR_INSURANCE_INFERENCE_SVC  |predict          |{                                    |1186.3138427734375   |
|                            |                                      |                             |                 |  "AGE": 61,                         |                     |
|                            |                       

In [ ]:
# Show all available columns and attributes in the inference table
session.sql(f"""
    SELECT *
    FROM TABLE(INFERENCE_TABLE({inference_table_ref}))
    LIMIT 5
""").show()

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TIMESTAMP"                 |"START_TIMESTAMP"  |"OBSERVED_TIMESTAMP"        |"TRACE"  |"RESOURCE"  |"RESOURCE_ATTRIBUTES"                               |"SCOPE"                      |"SCOPE_ATTRIBUTES"  |"RECORD_TYPE"  |"RECORD"  |"RECORD_ATTRIBUTES"                                 |"VALUE"    |"EXEMPLARS"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-03-03 16:47:39.786160  |NULL               |2026-03-03 

## 5. Create Task-Based Transform Pipeline

**Note:** `INFERENCE_TABLE()` is a table function, not a physical table — you cannot create a stream on it.
Instead, we use a **polling-based approach** with a watermark table that tracks the last-processed timestamp.
A scheduled task runs every 5 minutes, reads new inference records since the last watermark,
calls the `transform` function on the model, and stores results in `TRANSFORMATIONS_RESULTS`.

In [ ]:
# Create the table to store input data, predictions, and transformation results
session.sql(f"""
    CREATE OR REPLACE TABLE {DATABASE}.{SCHEMA}.TRANSFORMATIONS_RESULTS (
        -- Metadata
        INFERENCE_TIMESTAMP   TIMESTAMP_NTZ,
        QUERY_ID              VARCHAR,
        SERVICE_NAME          VARCHAR,
        FUNCTION_NAME         VARCHAR,
        TRANSFORM_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        -- Original input values
        CAR_AGE_INPUT               NUMBER,
        KILOMETERS_INPUT            NUMBER,
        ENGINE_SIZE_INPUT           FLOAT,
        ESTIMATED_CAR_VALUE_INPUT   FLOAT,
        AGE_INPUT                   NUMBER,
        YEARS_LICENSED_INPUT        NUMBER,
        CLAIMS_HISTORY_INPUT        NUMBER,
        CREDIT_SCORE_INPUT          NUMBER,
        RISK_SCORE_INPUT            FLOAT,
        AVG_CLAIMS_PER_YEAR_INPUT   FLOAT,
        TOTAL_POLICIES_INPUT        NUMBER,
        AVG_CAR_AGE_INPUT           FLOAT,
        AVG_KILOMETERS_INPUT        NUMBER,
        TOTAL_CAR_VALUE_INPUT       FLOAT,
        AVG_DEDUCTIBLE_INPUT        NUMBER,
        CAR_MAKE_INPUT              VARCHAR,
        CAR_MODEL_INPUT             VARCHAR,
        FUEL_TYPE_INPUT             VARCHAR,
        TRANSMISSION_INPUT          VARCHAR,
        COVERAGE_TYPE_INPUT         VARCHAR,
        GENDER_INPUT                VARCHAR,
        STATE_INPUT                 VARCHAR,
        -- Predicted premium from the predict endpoint
        PREDICTED_PREMIUM           FLOAT,
        -- Transformed (scaled/encoded) features from the transform endpoint
        CAR_AGE_TRANSFORMED               FLOAT,
        KILOMETERS_TRANSFORMED            FLOAT,
        ENGINE_SIZE_TRANSFORMED           FLOAT,
        ESTIMATED_CAR_VALUE_TRANSFORMED   FLOAT,
        AGE_TRANSFORMED                   FLOAT,
        YEARS_LICENSED_TRANSFORMED        FLOAT,
        CLAIMS_HISTORY_TRANSFORMED        FLOAT,
        CREDIT_SCORE_TRANSFORMED          FLOAT,
        RISK_SCORE_TRANSFORMED            FLOAT,
        AVG_CLAIMS_PER_YEAR_TRANSFORMED   FLOAT,
        TOTAL_POLICIES_TRANSFORMED        FLOAT,
        AVG_CAR_AGE_TRANSFORMED           FLOAT,
        AVG_KILOMETERS_TRANSFORMED        FLOAT,
        TOTAL_CAR_VALUE_TRANSFORMED       FLOAT,
        AVG_DEDUCTIBLE_TRANSFORMED        FLOAT,
        CAR_MAKE_ENCODED                  FLOAT,
        CAR_MODEL_ENCODED                 FLOAT,
        FUEL_TYPE_ENCODED                 FLOAT,
        TRANSMISSION_ENCODED              FLOAT,
        COVERAGE_TYPE_ENCODED             FLOAT,
        GENDER_ENCODED                    FLOAT,
        STATE_ENCODED                     FLOAT
    )
""").collect()

# Create a watermark table to track the last-processed inference timestamp
session.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK (
        WATERMARK_KEY    VARCHAR DEFAULT 'TRANSFORM_TASK',
        LAST_PROCESSED   TIMESTAMP_NTZ DEFAULT '1970-01-01'::TIMESTAMP_NTZ
    )
""").collect()

# Initialize the watermark if it doesn't exist
session.sql(f"""
    MERGE INTO {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK t
    USING (SELECT 'TRANSFORM_TASK' AS WATERMARK_KEY) s
    ON t.WATERMARK_KEY = s.WATERMARK_KEY
    WHEN NOT MATCHED THEN INSERT (WATERMARK_KEY, LAST_PROCESSED)
        VALUES ('TRANSFORM_TASK', '1970-01-01'::TIMESTAMP_NTZ)
""").collect()

print(f"Created table: {DATABASE}.{SCHEMA}.TRANSFORMATIONS_RESULTS")
print(f"Created table: {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK")

Created table: CC_ML_INSURANCE.CAR_PRICING.TRANSFORMATIONS_RESULTS
Created table: CC_ML_INSURANCE.CAR_PRICING.INFERENCE_WATERMARK


In [ ]:
# Create a stored procedure that:
# 1. Reads the last-processed watermark
# 2. Queries INFERENCE_TABLE() for new records since that watermark
# 3. Extracts input values, predicted premium, and calls TRANSFORM
# 4. Inserts everything into TRANSFORMATIONS_RESULTS
# 5. Updates the watermark

PROC_NAME = "PROCESS_NEW_INFERENCES_SP"

session.sql(f"""
    CREATE OR REPLACE PROCEDURE {DATABASE}.{SCHEMA}.{PROC_NAME}()
    RETURNS VARCHAR
    LANGUAGE SQL
    EXECUTE AS CALLER
    AS
    $$
    DECLARE
        last_ts TIMESTAMP_NTZ;
        new_max_ts TIMESTAMP_NTZ;
        rows_inserted INTEGER DEFAULT 0;
    BEGIN
        -- Get the last-processed watermark
        SELECT LAST_PROCESSED INTO :last_ts
        FROM {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK
        WHERE WATERMARK_KEY = 'TRANSFORM_TASK';

        -- Get the max timestamp of new records
        SELECT MAX(TIMESTAMP) INTO :new_max_ts
        FROM TABLE(INFERENCE_TABLE('{DATABASE}.{SCHEMA}.{MODEL_NAME}'))
        WHERE TIMESTAMP > :last_ts
          AND LOWER(RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING) = 'predict';

        -- If there are new records, process them
        IF (:new_max_ts IS NOT NULL) THEN
            INSERT INTO {DATABASE}.{SCHEMA}.TRANSFORMATIONS_RESULTS (
                -- Metadata
                INFERENCE_TIMESTAMP, QUERY_ID, SERVICE_NAME, FUNCTION_NAME,
                -- Input values
                CAR_AGE_INPUT, KILOMETERS_INPUT, ENGINE_SIZE_INPUT, ESTIMATED_CAR_VALUE_INPUT,
                AGE_INPUT, YEARS_LICENSED_INPUT, CLAIMS_HISTORY_INPUT, CREDIT_SCORE_INPUT,
                RISK_SCORE_INPUT, AVG_CLAIMS_PER_YEAR_INPUT, TOTAL_POLICIES_INPUT, AVG_CAR_AGE_INPUT,
                AVG_KILOMETERS_INPUT, TOTAL_CAR_VALUE_INPUT, AVG_DEDUCTIBLE_INPUT,
                CAR_MAKE_INPUT, CAR_MODEL_INPUT, FUEL_TYPE_INPUT, TRANSMISSION_INPUT,
                COVERAGE_TYPE_INPUT, GENDER_INPUT, STATE_INPUT,
                -- Predicted premium
                PREDICTED_PREMIUM,
                -- Transformed features
                CAR_AGE_TRANSFORMED, KILOMETERS_TRANSFORMED, ENGINE_SIZE_TRANSFORMED, ESTIMATED_CAR_VALUE_TRANSFORMED,
                AGE_TRANSFORMED, YEARS_LICENSED_TRANSFORMED, CLAIMS_HISTORY_TRANSFORMED, CREDIT_SCORE_TRANSFORMED,
                RISK_SCORE_TRANSFORMED, AVG_CLAIMS_PER_YEAR_TRANSFORMED, TOTAL_POLICIES_TRANSFORMED, AVG_CAR_AGE_TRANSFORMED,
                AVG_KILOMETERS_TRANSFORMED, TOTAL_CAR_VALUE_TRANSFORMED, AVG_DEDUCTIBLE_TRANSFORMED,
                CAR_MAKE_ENCODED, CAR_MODEL_ENCODED, FUEL_TYPE_ENCODED, TRANSMISSION_ENCODED,
                COVERAGE_TYPE_ENCODED, GENDER_ENCODED, STATE_ENCODED
            )
            SELECT
                -- Metadata
                inf.TIMESTAMP,
                inf.RESOURCE_ATTRIBUTES:"snow.query.id"::STRING,
                inf.RESOURCE_ATTRIBUTES:"snow.service.name"::STRING,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING,
                -- Input values
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_AGE"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.KILOMETERS"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.ENGINE_SIZE"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.ESTIMATED_CAR_VALUE"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.YEARS_LICENSED"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.CLAIMS_HISTORY"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.CREDIT_SCORE"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.RISK_SCORE"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CLAIMS_PER_YEAR"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_POLICIES"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CAR_AGE"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_KILOMETERS"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_CAR_VALUE"::FLOAT,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_DEDUCTIBLE"::NUMBER,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MAKE"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MODEL"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.FUEL_TYPE"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.TRANSMISSION"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.COVERAGE_TYPE"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.GENDER"::VARCHAR,
                inf.RECORD_ATTRIBUTES:"snow.model_serving.request.data.STATE"::VARCHAR,
                -- Predicted premium
                inf.RECORD_ATTRIBUTES:"snow.model_serving.response.data.PREDICTED_PREMIUM"::FLOAT,
                -- Transformed features
                inf.t:"CAR_AGE"::FLOAT,
                inf.t:"KILOMETERS"::FLOAT,
                inf.t:"ENGINE_SIZE"::FLOAT,
                inf.t:"ESTIMATED_CAR_VALUE"::FLOAT,
                inf.t:"AGE"::FLOAT,
                inf.t:"YEARS_LICENSED"::FLOAT,
                inf.t:"CLAIMS_HISTORY"::FLOAT,
                inf.t:"CREDIT_SCORE"::FLOAT,
                inf.t:"RISK_SCORE"::FLOAT,
                inf.t:"AVG_CLAIMS_PER_YEAR"::FLOAT,
                inf.t:"TOTAL_POLICIES"::FLOAT,
                inf.t:"AVG_CAR_AGE"::FLOAT,
                inf.t:"AVG_KILOMETERS"::FLOAT,
                inf.t:"TOTAL_CAR_VALUE"::FLOAT,
                inf.t:"AVG_DEDUCTIBLE"::FLOAT,
                inf.t:"CAR_MAKE_ENCODED"::FLOAT,
                inf.t:"CAR_MODEL_ENCODED"::FLOAT,
                inf.t:"FUEL_TYPE_ENCODED"::FLOAT,
                inf.t:"TRANSMISSION_ENCODED"::FLOAT,
                inf.t:"COVERAGE_TYPE_ENCODED"::FLOAT,
                inf.t:"GENDER_ENCODED"::FLOAT,
                inf.t:"STATE_ENCODED"::FLOAT
            FROM (
                SELECT
                    *,
                    {DATABASE}.{SCHEMA}.{MODEL_NAME}!TRANSFORM(
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_AGE"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.KILOMETERS"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.ENGINE_SIZE"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.ESTIMATED_CAR_VALUE"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.YEARS_LICENSED"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.CLAIMS_HISTORY"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.CREDIT_SCORE"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.RISK_SCORE"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CLAIMS_PER_YEAR"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_POLICIES"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_CAR_AGE"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_KILOMETERS"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_CAR_VALUE"::FLOAT,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_DEDUCTIBLE"::NUMBER,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MAKE"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.CAR_MODEL"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.FUEL_TYPE"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.TRANSMISSION"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.COVERAGE_TYPE"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.GENDER"::VARCHAR,
                        RECORD_ATTRIBUTES:"snow.model_serving.request.data.STATE"::VARCHAR
                    ) AS t
                FROM TABLE(INFERENCE_TABLE('{DATABASE}.{SCHEMA}.{MODEL_NAME}'))
                WHERE TIMESTAMP > :last_ts
                  AND TIMESTAMP <= :new_max_ts
                  AND LOWER(RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING) = 'predict'
            ) inf;

            rows_inserted := SQLROWCOUNT;

            -- Update the watermark
            UPDATE {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK
            SET LAST_PROCESSED = :new_max_ts
            WHERE WATERMARK_KEY = 'TRANSFORM_TASK';

            RETURN 'Processed ' || :rows_inserted || ' rows. Watermark updated to ' || :new_max_ts::VARCHAR;
        ELSE
            RETURN 'No new inference records to process.';
        END IF;
    END;
    $$
""").collect()

print(f"Created procedure: {DATABASE}.{SCHEMA}.{PROC_NAME}")

Created procedure: CC_ML_INSURANCE.CAR_PRICING.PROCESS_NEW_INFERENCES_SP


In [ ]:
# Create a task that runs every 5 minutes and calls the stored procedure
TASK_NAME = "TRANSFORM_INFERENCE_TASK"

session.sql(f"""
    CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.{TASK_NAME}
        WAREHOUSE = 'COMPUTE_WH'
        SCHEDULE = '5 MINUTE'
    AS
        CALL {DATABASE}.{SCHEMA}.{PROC_NAME}()
""").collect()

print(f"Created task: {DATABASE}.{SCHEMA}.{TASK_NAME}")
print("Schedule: every 5 minutes, calls the stored procedure to poll for new inference records")

Created task: CC_ML_INSURANCE.CAR_PRICING.TRANSFORM_INFERENCE_TASK
Schedule: every 5 minutes, calls the stored procedure to poll for new inference records


In [ ]:
# Resume the task (tasks are created in suspended state by default)
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.{TASK_NAME} RESUME").collect()

print(f"Task {TASK_NAME} resumed")

# Verify the task
task_info = session.sql(f"SHOW TASKS LIKE '{TASK_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
for t in task_info:
    print(f"  Task: {t['name']}, State: {t['state']}, Schedule: {t['schedule']}")

Task TRANSFORM_INFERENCE_TASK resumed
  Task: TRANSFORM_INFERENCE_TASK, State: started, Schedule: 5 MINUTE


## 6. Verify the Pipeline

Run a second batch of predictions to generate new inference records, then check that the task processes them.

In [ ]:
# Run a second, smaller batch to generate new inference table entries
second_batch = enriched_df.select(input_cols).limit(10)

predictions_2 = mv.run(
    second_batch,
    function_name="predict",
    service_name=SERVICE_NAME
)

print("Second batch predictions (10 rows) - these will be picked up by the task:")
predictions_2.show()

In [ ]:
# You can manually call the procedure to test without waiting 5 minutes
result = session.sql(f"CALL {DATABASE}.{SCHEMA}.{PROC_NAME}()").collect()
print(f"Procedure result: {result[0][0]}")

Procedure result: Processed 20 rows. Watermark updated to 2026-03-03 16:47:39.786


In [ ]:
import time

# Wait a moment for the task execution to complete
#time.sleep(10)

# Check the transformations results table
results_df = session.sql(f"""
    SELECT *
    FROM {DATABASE}.{SCHEMA}.TRANSFORMATIONS_RESULTS
    ORDER BY TRANSFORM_TIMESTAMP DESC
    LIMIT 20
""")

print(f"Transformation results ({results_df.count()} total rows):")
results_df.show()

Transformation results (20 total rows):
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Check task execution history
task_history = session.sql(f"""
    SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        TASK_NAME => '{TASK_NAME}',
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ))
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 10
""")

print("Task execution history:")
task_history.show()

# Check the current watermark
watermark = session.sql(f"""
    SELECT * FROM {DATABASE}.{SCHEMA}.INFERENCE_WATERMARK
""")
print("\nCurrent watermark:")
watermark.show()

Task execution history:
--------------------------------------------------------------------------------------------------------------------------------
|"NAME"                    |"STATE"    |"SCHEDULED_TIME"                  |"COMPLETED_TIME"                  |"ERROR_MESSAGE"  |
--------------------------------------------------------------------------------------------------------------------------------
|TRANSFORM_INFERENCE_TASK  |SCHEDULED  |2026-03-03 08:57:04.342000-08:00  |NULL                              |NULL             |
|TRANSFORM_INFERENCE_TASK  |SUCCEEDED  |2026-03-03 08:43:52.764000-08:00  |2026-03-03 08:43:54.773000-08:00  |NULL             |
|TRANSFORM_INFERENCE_TASK  |SUCCEEDED  |2026-03-03 08:38:52.764000-08:00  |2026-03-03 08:38:54.031000-08:00  |NULL             |
|TRANSFORM_INFERENCE_TASK  |SUCCEEDED  |2026-03-03 08:33:52.764000-08:00  |2026-03-03 08:33:55.540000-08:00  |NULL             |
|TRANSFORM_INFERENCE_TASK  |SUCCEEDED  |2026-03-03 08:28:52.764000-08:00 

## 7. Cleanup

Suspend the task to avoid unnecessary warehouse consumption.

In [ ]:
# Suspend the task to stop scheduled execution
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.{TASK_NAME} SUSPEND").collect()

print(f"Task {TASK_NAME} suspended")

# Verify
task_info = session.sql(f"SHOW TASKS LIKE '{TASK_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
for t in task_info:
    print(f"  Task: {t['name']}, State: {t['state']}")

Task TRANSFORM_INFERENCE_TASK suspended
  Task: TRANSFORM_INFERENCE_TASK, State: suspended
